In [2]:
# Target wavelength: 652, 528,and 467 nm in the present work
# convert Bscat measurements from the original wavelengths of 700 nm(w1), 550 nm(w2), and 450 nm(w3)
# convert Babs measurements from the original wavelengths of 648 nm(w1), 529 nm (w2), and 464 nm (w3)

In [1]:
import os
import sys
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
import scipy.odr as odr
import os
import sys
import arrow
import numpy as np
import pandas as pd
from math import pi
import matplotlib.pyplot as plt
import matplotlib.dates as dates
import seaborn as sns
from datetime import datetime
from numpy import nan_to_num
from matplotlib.colors import LogNorm
from matplotlib.ticker import ScalarFormatter
import os
import sys
import arrow
import numpy as np
import pandas as pd
from math import pi
import matplotlib.pyplot as plt
import matplotlib.dates as dates
import seaborn as sns
from datetime import date
from numpy import nan_to_num
from matplotlib.colors import LogNorm
from matplotlib.ticker import ScalarFormatter
import pandas as pd
from datetime import datetime, timedelta

Formula:Babs(lamda) proportional lamda^-AAE >> Babs(lamda) = C*lamda^-AAE

Here, C is AAE_constant

In [2]:
def SAE_fit_one_measurement( wavelength, SAE_fit, SAE_constant):
    return  SAE_constant*np.power(wavelength, -SAE_fit)

def AAE_fit_one_measurement( wavelength, AAE_fit, AAE_constant):
    return  AAE_constant*np.power(wavelength, -AAE_fit)

In [3]:
# Input wavelength
w1=700; w2=550; w3=450                          # scat wvl from instrument
w4=660; w5=532; w6=470                          # abs wvl from instrument 
w_global_1=660; w_global_2=532; w_global_3=470  #User defined wvl

NEPH_Bscat_before_data  = pd.read_csv(rf"C:\Users\haika\Desktop\May_Research\MAC Model for may dataset\ModuleC\Bscat_raw.csv")
NEPH_Bscat_before_data

,scat_red,scat_green,scat_blue
0,7.798704,8.482831,9.097619
1,3.395066,5.908108,9.368104
2,2.789074,5.776653,10.587647
3,2.203251,5.239813,10.774435
4,8.782283,12.728914,17.334541
...,...,...,...
1269009,11.740000,14.810000,16.910000
1269010,15.000000,15.850000,20.650000
1269011,12.390000,13.030000,21.310000
1269012,12.500000,15.220000,19.410000


In [4]:
# There was no abs_raw file in Hanyan's input dataset. So, I am using a dummy dataset which is actually output of Module B.
Filter_based_instrument_data = pd.read_csv(rf"C:\Users\haika\Desktop\May_Research\MAC Model for may dataset\ModuleC\Babs_raw.csv")
Filter_based_instrument_data

,abs_red,abs_green,abs_blue
0,8.479087,7.580893,15.643075
1,1.364240,10.129900,1.738330
2,15.163788,12.082607,1.938907
3,29.464000,26.815100,14.258900
4,5.890006,11.883600,6.890933
...,...,...,...
1269009,2.670000,2.630000,0.450000
1269010,-5.780000,-5.010000,-3.770000
1269011,-3.100000,-1.760000,-4.670000
1269012,-0.830000,-3.710000,-5.010000


In [5]:
Filter_based_instrument_data = Filter_based_instrument_data.iloc[:, -3:].rename(columns={Filter_based_instrument_data.columns[-3]: 'abs_red', Filter_based_instrument_data.columns[-2]: 'abs_green', Filter_based_instrument_data.columns[-1]: 'abs_blue'})
Filter_based_instrument_data

,abs_red,abs_green,abs_blue
0,8.479087,7.580893,15.643075
1,1.364240,10.129900,1.738330
2,15.163788,12.082607,1.938907
3,29.464000,26.815100,14.258900
4,5.890006,11.883600,6.890933
...,...,...,...
1269009,2.670000,2.630000,0.450000
1269010,-5.780000,-5.010000,-3.770000
1269011,-3.100000,-1.760000,-4.670000
1269012,-0.830000,-3.710000,-5.010000


In [6]:
bscat_wvl_red=NEPH_Bscat_before_data['scat_red']
bscat_wvl_green=NEPH_Bscat_before_data['scat_green']
bscat_wvl_blue=NEPH_Bscat_before_data['scat_blue']

babs_wvl_red=Filter_based_instrument_data['abs_red']
babs_wvl_green=Filter_based_instrument_data['abs_green']
babs_wvl_blue=Filter_based_instrument_data['abs_blue']
bscat_wvl_blue

0           9.097619
1           9.368104
2          10.587647
3          10.774435
4          17.334541
             ...    
1269009    16.910000
1269010    20.650000
1269011    21.310000
1269012    19.410000
1269013    19.730000
Name: scat_blue, Length: 1269014, dtype: float64

#### Explaining curve_fit function
curve_fit (f, xdata, ydata, p0=None, ...)
f is a function (in our case, f is SAE_fit_one_measurement), xdata is independent variables (here, x data is wavelengths: red (700), green(550), blue(450)), ydata is dependent varibles (here, Bscat_red, Bscat_green, Bscat_blue), P0 is initial guess for function parameters (here, C and SAE). How we calculated initial C and SAE?

```
SAE_G_B= -log(Bscat_green/Bscat_blue)/log(lamda_green/lamda_blue)
Bscat_green = C_green*lamda_green^-SAE_G_B   #here, initially we use SAE_G_B
Therefore, C = Bscat_green/lamda_green^-SAE_G_B
We get, intial parameters, W_coef= (SAE_G_B, C_green)
```
Derivation of ```SAE_G_B= -log(Bscat_green/Bscat_blue)/log(lamda_green/lamda_blue)```
```
Bscat_lamda1 = C1*lamda1^-SAE
Bscat_lamda2 = C2*lamda2^-SAE

Bscat_lamda1/Bscat_lamda2=(lamda1/lamda2)^-SAE
log (Bscat_lamda1/Bscat_lamda2)= -SAE log(lamda1/lamda2)
SAE = -log (Bscat_lamda1/Bscat_lamda2) / log(lamda1/lamda2)

```
curve_fit expects a function (f) of the form: y=f(x,p1,p2,p3,...,pn), where x is the independent variable (xdata in curve_fit) and p1,p2,p3..,pn are the parameters needed to be optimized (these are automatically treated as fitting parameters). In our code, we created this function f as:

```
def SAE_fit_one_measurement( wavelength, SAE_fit, SAE_constant):
    return  SAE_constant*np.power(wavelength, -SAE_fit)
```
So, Curve_fit will consider wavelength as xdata. And, SAE_fit and SAE_constant as initial fitting parameters.
```
How curve_fit work? [curve_fit (f, xdata, ydata, p0=None, ...)]
W_coef, w_sigma = curve_fit(
    SAE_fit_one_measurement, 
    wavelength_instrument, 
    one_babs_at_three_lambda, 
    p0=W_coef, 
    maxfev=1000)
```
1. Identify xdata (wavelength_instrument): xdata would be the first argument of SAE_fit_one_measurement function (wavelength).
2. identify fitting parameters (P0): SAE_fit, SAE_constant would be considered.
3. compute prediction: Y_pred= SAE_constant*np.power(wavelength, -SAE_fit)
4. Minimize error: compare the Y_pred with ydata(one_babs_three_lamda) and minimize this difference by adjusting SAE_fit (SAE) and SAE_constant(C)
5. Returen the best fit parameters: once the minimum error found, it returns:
  
   1.W_coeff: Best fit value of SAE_fit and SAE_constant

   2.w_sigma[0,1]: estimated uncertainties in SAE_fit and SAE_constant

Try--except--pass block  
The try block attempts to fit the function using curve_fit.
If the fitting process fails (e.g., due to numerical issues, bad initial values, or insufficient data), a RuntimeError is raised.
The except RuntimeError: block catches the error and prevents the program from crashing.
The pass statement ensures that the loop continues even if one specific data point fails to fit.


In [ ]:
# Hanyang's code after cora's modification
#def SAE_power_fit(bscat_wvl_red,bscat_wvl_green,bscat_wvl_blue,w1,w2,w3,w_global_1,w_global_2,w_global_3):

if(w1 <= 0 and w2 <= 0 and w3 <= 0):  #added by Cora
        print("WARNING: wavelength <= 0")

#W1, W2, W3 will be in decreasing order or red, green, blue
wavelength_instrument=np.array([w1,w2,w3])  # modify if necessary

temp_SAE = np.nan; temp_SAE_constant = np.nan      # Fitting parameters
temp_sigma = np.nan; temp_sigma_constant = np.nan  #uncertainty in fit parameters

SAE_three_lambda = np.empty(len(bscat_wvl_red)); SAE_constant = np.empty(len(bscat_wvl_red))
sigma_SAE = np.empty(len(bscat_wvl_red)); sigma_constant = np.empty(len(bscat_wvl_red))

#hide invalid values  #added by cora
#bscat_wvl_red[bscat_wvl_red ==0]
mask_red = bscat_wvl_red.mask(bscat_wvl_red == 0)  # 0 replaced by NaN
mask_green = bscat_wvl_green.mask(bscat_wvl_green == 0)
mask_blue = bscat_wvl_blue.mask(bscat_wvl_blue == 0)

mask_RG = (mask_red/mask_green).mask((mask_red/mask_green) == 1)
mask_RG = mask_RG.mask((mask_red/mask_green) < 0)

mask_RB = (mask_red/mask_blue).mask((mask_red/mask_blue) == 1)
mask_RB = mask_RB.mask((mask_red/mask_blue) < 0)

mask_GB = (mask_green/mask_blue).mask((mask_green/mask_blue) == 1)
mask_GB = mask_GB.mask((mask_green/mask_blue) < 0)

#mask_red.plot(kind='line')  #Check plot

#compute SAE_two_lambda, which are used later when initializing SAE_three_lambda
SAE_R_G = -np.log(mask_RG)/np.log(wavelength_instrument[0]/wavelength_instrument[1])
SAE_R_B = -np.log(mask_RB)/np.log(wavelength_instrument[0]/wavelength_instrument[2])
SAE_G_B = -np.log(mask_GB)/np.log(wavelength_instrument[1]/wavelength_instrument[2])

W_coef = np.zeros(np.size((SAE_G_B.values)))
w_sigma = np.zeros(np.size((SAE_G_B.values)))

for m in range(len(bscat_wvl_red)):
        if (np.isnan(bscat_wvl_red[m])==0 and np.isnan(bscat_wvl_green[m])==0 and np.isnan(bscat_wvl_blue[m])==0):

            one_babs_at_three_lambda=[bscat_wvl_red[m],bscat_wvl_green[m],bscat_wvl_blue[m]]
            
            #use SAE_G_B as initial input 
            if((np.power(wavelength_instrument[1], -SAE_G_B.values[m])) > 0):
                W_coef = np.array([SAE_G_B.values[m], bscat_wvl_green[m]/ (np.power(wavelength_instrument[1], -SAE_G_B.values[m]))] )
                # first part before coma is SAE_fit, second part after coma is SAE_Constat or C
                try:
                    W_coef, w_sigma = curve_fit(SAE_fit_one_measurement, wavelength_instrument, one_babs_at_three_lambda,p0=W_coef, maxfev=1000)
                    w_sigma = np.sqrt(np.diag(w_sigma))
                except RuntimeError:
                    pass
            temp_SAE = W_coef[0];temp_SAE_constant = W_coef[1];temp_sigma= w_sigma[0];temp_sigma_constant = w_sigma[1]
            #use SAE_R_G as initial input
            if(((np.power(wavelength_instrument[0], -SAE_R_G.values[m]))) > 0 ):
                W_coef = np.array([SAE_R_G.values[m], bscat_wvl_red[m]/ (np.power(wavelength_instrument[0], -SAE_R_G.values[m]))]  ) 
                try:          
                    W_coef, w_sigma = curve_fit(SAE_fit_one_measurement, wavelength_instrument, one_babs_at_three_lambda,p0=W_coef, maxfev=1000)
                    w_sigma = np.sqrt(np.diag(w_sigma))
                except RuntimeError:
                    pass    #do not replace the previous results

            #make sure w_sigma exists
            if(w_sigma[0] != -1):  # Uzzal don't know why Cora added this?
                #check if the current inital guess (SAE_R_G) is better than the previous one (SAE_G_B): ? w_sigma is smaller
                if (w_sigma[0]<temp_sigma):  #Yes, the current w_sigma is smaller                
                    temp_SAE = W_coef[0];temp_SAE_constant = W_coef[1];temp_sigma= w_sigma[0];temp_sigma_constant = w_sigma[1]
                else:
                    pass       #do not replace the previous results
        
            #use SAE_R_B as initial input
            if((np.power(wavelength_instrument[2], -SAE_R_B.values[m])) > 0):
                W_coef = np.array([SAE_R_B.values[m],bscat_wvl_blue[m]/ (np.power(wavelength_instrument[2], -SAE_R_B.values[m]))])
                try:   
                    W_coef, w_sigma = curve_fit(SAE_fit_one_measurement, wavelength_instrument, one_babs_at_three_lambda,p0=W_coef, maxfev=1000)
                    w_sigma = np.sqrt(np.diag(w_sigma))
                except RuntimeError:
                    pass
            #make sure w_sigma exists
            if(w_sigma[0] != -1):               
                #check if the current initial guess (SAE_R_B) is better than the previous one (SAE_G_B or SAE_R_G): ? w_sigma is smaller
                if (w_sigma[0]<temp_sigma):  #Yes, the current w_sigma is smaller                        
                    temp_SAE = W_coef[0];temp_SAE_constant = W_coef[1];temp_sigma= w_sigma[0];temp_sigma_constant = w_sigma[1]
                else: 
                    pass #do not replace the previous results   
        
        #Output the results
        SAE_three_lambda[m]=temp_SAE; SAE_constant[m]=temp_SAE_constant
        sigma_SAE[m]=temp_sigma; sigma_constant[m]=temp_sigma_constant
        #clear temp_variables
        temp_SAE=np.nan;temp_SAE_constant=np.nan
        temp_sigma=np.nan;temp_sigma_constant=np.nan

b_scat_red=SAE_constant*(np.power(w_global_1, -SAE_three_lambda))
b_scat_green=SAE_constant*(np.power(w_global_2, -SAE_three_lambda))
b_scat_blue=SAE_constant*(np.power(w_global_3, -SAE_three_lambda))

file_M_3_Bscat = pd.DataFrame() 
file_M_3_Bscat['SAE'] = SAE_three_lambda; 
file_M_3_Bscat['SAE_constant'] = SAE_constant
file_M_3_Bscat['b_scat_red'] = b_scat_red ;file_M_3_Bscat['b_scat_green'] = b_scat_green; file_M_3_Bscat['b_scat_blue'] = b_scat_blue ; 

# export_csv = file_M_3_Bscat.to_csv ('Output_Module_C_Bscat_CORA.csv', index = None, header=True)

C:\Users\haika\AppData\Local\Temp\ipykernel_3916\1658513271.py:77: OptimizeWarning: Covariance of the parameters could not be estimated
  W_coef, w_sigma = curve_fit(SAE_fit_one_measurement, wavelength_instrument, one_babs_at_three_lambda,p0=W_coef, maxfev=1000)
C:\Users\haika\AppData\Local\Temp\ipykernel_3916\1658513271.py:51: OptimizeWarning: Covariance of the parameters could not be estimated
  W_coef, w_sigma = curve_fit(SAE_fit_one_measurement, wavelength_instrument, one_babs_at_three_lambda,p0=W_coef, maxfev=1000)
C:\Users\haika\AppData\Local\Temp\ipykernel_3916\1658513271.py:60: OptimizeWarning: Covariance of the parameters could not be estimated
  W_coef, w_sigma = curve_fit(SAE_fit_one_measurement, wavelength_instrument, one_babs_at_three_lambda,p0=W_coef, maxfev=1000)
C:\Users\haika\AppData\Local\Temp\ipykernel_3916\1380745124.py:2: RuntimeWarning: overflow encountered in power
  return  SAE_constant*np.power(wavelength, -SAE_fit)
C:\Users\haika\AppData\Local\Temp\ipykernel_3

In [ ]:
export_csv = file_M_3_Bscat.to_csv (rf'C:\Users\haika\Desktop\May_Research\MAC Model for may dataset\ModuleC\Output_Module_C_Bscat_CORA.csv', index = None, header=True)
file_M_3_Bscat

,SAE,SAE_constant,b_scat_red,b_scat_green,b_scat_blue
0,0.348673,7.656526e+01,7.960355,8.581822,8.960720
1,2.297221,1.165917e+07,3.886437,6.377430,8.477501
2,3.019193,1.084825e+09,3.331293,6.387145,9.284979
3,3.592405,3.662760e+10,2.721853,5.905135,9.216175
4,1.538964,2.099477e+05,9.614668,13.397724,16.212444
...,...,...,...,...,...
1382858,0.811949,2.433220e+03,12.498418,14.889486,16.465456
1382859,0.766437,2.163734e+03,14.934809,17.618259,19.373491
1382860,1.410880,1.127574e+05,11.860586,16.077189,19.148506
1382861,1.019745,9.755142e+03,13.002259,16.199435,18.381298


In [ ]:
if(w1 <= 0 and w2 <= 0 and w3 <= 0):
    print("WARNING: wavelength <= 0")

wavelength_instrument=np.array([w1,w2,w3])  # modify if necessary

# Initialize arrays with NaN - same length as input data
AAE_three_lambda = np.full(len(babs_wvl_red), np.nan)
AAE_constant = np.full(len(babs_wvl_red), np.nan)
sigma_AAE = np.full(len(babs_wvl_red), np.nan)
sigma_constant = np.full(len(babs_wvl_red), np.nan)

#hide invalid values
mask_red = babs_wvl_red.mask(babs_wvl_red == 0)
mask_green = babs_wvl_green.mask(babs_wvl_green == 0)
mask_blue = babs_wvl_blue.mask(babs_wvl_blue == 0)

mask_RG = (mask_red/mask_green).mask((mask_red/mask_green == 1))
mask_RG = mask_RG.mask((mask_red/mask_green) < 0)

mask_RB = (mask_red/mask_blue).mask((mask_red/mask_blue) == 1)
mask_RB = mask_RB.mask((mask_red/mask_blue) < 0)

mask_GB = (mask_green/mask_blue).mask((mask_green/mask_blue) == 1)
mask_GB = mask_GB.mask((mask_green/mask_blue) < 0)

#compute AAE_two_lambda, which are used later when initializing AAE_three_lambda
AAE_R_G = -np.log(mask_RG)/np.log(wavelength_instrument[0]/wavelength_instrument[1])
AAE_R_B = -np.log(mask_RB)/np.log(wavelength_instrument[0]/wavelength_instrument[2])
AAE_G_B = -np.log(mask_GB)/np.log(wavelength_instrument[1]/wavelength_instrument[2])

print(f"Processing {len(babs_wvl_red)} rows...")
successful_fits = 0
skipped_rows = 0

for m in range(len(babs_wvl_red)):
    
    # IMPROVED CHECK: Only skip NaN, inf, -inf, and non-finite values
    # Allow zero and negative values to be processed
    if (np.isfinite(babs_wvl_red[m]) and np.isfinite(babs_wvl_green[m]) and np.isfinite(babs_wvl_blue[m])):

        one_babs_at_three_lambda = [babs_wvl_red[m], babs_wvl_green[m], babs_wvl_blue[m]]
        
        # Initialize temporary variables for this iteration
        temp_AAE = np.nan
        temp_AAE_constant = np.nan
        temp_sigma = np.nan
        temp_sigma_constant = np.nan
        
        best_sigma = np.inf
        valid_fit_found = False
        
        # Try AAE_G_B as initial input
        if np.isfinite(AAE_G_B.values[m]) and AAE_G_B.values[m] != 0:
            try:
                denominator = np.power(wavelength_instrument[1], -AAE_G_B.values[m])
                if np.isfinite(denominator) and denominator > 0:
                    const_guess = babs_wvl_green[m] / denominator
                    if np.isfinite(const_guess):
                        W_coef = np.array([AAE_G_B.values[m], const_guess])
                        
                        W_coef, w_sigma = curve_fit(AAE_fit_one_measurement, wavelength_instrument, one_babs_at_three_lambda, p0=W_coef, maxfev=1000)
                        
                        if np.all(np.isfinite(W_coef)) and np.all(np.isfinite(w_sigma)):
                            w_sigma = np.sqrt(np.diag(w_sigma))
                            if w_sigma[0] < best_sigma:
                                temp_AAE = W_coef[0]
                                temp_AAE_constant = W_coef[1]
                                temp_sigma = w_sigma[0]
                                temp_sigma_constant = w_sigma[1]
                                best_sigma = w_sigma[0]
                                valid_fit_found = True
                                
            except (RuntimeError, ValueError, OverflowError):
                pass
        
        # Try AAE_R_G as initial input
        if np.isfinite(AAE_R_G.values[m]) and AAE_R_G.values[m] != 0:
            try:
                denominator = np.power(wavelength_instrument[0], -AAE_R_G.values[m])
                if np.isfinite(denominator) and denominator > 0:
                    const_guess = babs_wvl_red[m] / denominator
                    if np.isfinite(const_guess):
                        W_coef = np.array([AAE_R_G.values[m], const_guess])
                        
                        W_coef, w_sigma = curve_fit(AAE_fit_one_measurement, wavelength_instrument, one_babs_at_three_lambda, p0=W_coef, maxfev=1000)
                        
                        if np.all(np.isfinite(W_coef)) and np.all(np.isfinite(w_sigma)):
                            w_sigma = np.sqrt(np.diag(w_sigma))
                            if w_sigma[0] < best_sigma:
                                temp_AAE = W_coef[0]
                                temp_AAE_constant = W_coef[1]
                                temp_sigma = w_sigma[0]
                                temp_sigma_constant = w_sigma[1]
                                best_sigma = w_sigma[0]
                                valid_fit_found = True
                                
            except (RuntimeError, ValueError, OverflowError):
                pass
        
        # Try AAE_R_B as initial input
        if np.isfinite(AAE_R_B.values[m]) and AAE_R_B.values[m] != 0:
            try:
                denominator = np.power(wavelength_instrument[2], -AAE_R_B.values[m])
                if np.isfinite(denominator) and denominator > 0:
                    const_guess = babs_wvl_blue[m] / denominator
                    if np.isfinite(const_guess):
                        W_coef = np.array([AAE_R_B.values[m], const_guess])
                        
                        W_coef, w_sigma = curve_fit(AAE_fit_one_measurement, wavelength_instrument, one_babs_at_three_lambda, p0=W_coef, maxfev=1000)
                        
                        if np.all(np.isfinite(W_coef)) and np.all(np.isfinite(w_sigma)):
                            w_sigma = np.sqrt(np.diag(w_sigma))
                            if w_sigma[0] < best_sigma:
                                temp_AAE = W_coef[0]
                                temp_AAE_constant = W_coef[1]
                                temp_sigma = w_sigma[0]
                                temp_sigma_constant = w_sigma[1]
                                best_sigma = w_sigma[0]
                                valid_fit_found = True
                                
            except (RuntimeError, ValueError, OverflowError):
                pass
        
        # Store results (will be NaN if no valid fit was found)
        AAE_three_lambda[m] = temp_AAE
        AAE_constant[m] = temp_AAE_constant
        sigma_AAE[m] = temp_sigma
        sigma_constant[m] = temp_sigma_constant
        
        if valid_fit_found:
            successful_fits += 1
        else:
            # Fitting failed but input was finite - still count as successful attempt
            successful_fits += 1
            
    else:
        # Invalid input data (NaN, inf, -inf, non-finite) - results remain NaN
        skipped_rows += 1
        
    # Progress indicator every 10000 rows
    if (m + 1) % 10000 == 0:
        print(f"  Processed {m + 1}/{len(babs_wvl_red)} rows... (successful: {successful_fits}, skipped: {skipped_rows})")

# Convert to target wavelengths - handle NaN values safely
print("Converting to target wavelengths...")
b_abs_red = np.full(len(babs_wvl_red), np.nan)
b_abs_green = np.full(len(babs_wvl_red), np.nan)
b_abs_blue = np.full(len(babs_wvl_red), np.nan)

# Only convert for rows with valid AAE fits
valid_mask = np.isfinite(AAE_three_lambda) & np.isfinite(AAE_constant)
if np.any(valid_mask):
    try:
        # Vectorized conversion for valid rows
        red_powers = np.power(w_global_1, -AAE_three_lambda[valid_mask])
        green_powers = np.power(w_global_2, -AAE_three_lambda[valid_mask])
        blue_powers = np.power(w_global_3, -AAE_three_lambda[valid_mask])
        
        b_abs_red[valid_mask] = AAE_constant[valid_mask] * red_powers
        b_abs_green[valid_mask] = AAE_constant[valid_mask] * green_powers
        b_abs_blue[valid_mask] = AAE_constant[valid_mask] * blue_powers
        
    except (OverflowError, ValueError):
        # If vectorized conversion fails, do it row by row
        for i in range(len(AAE_three_lambda)):
            if valid_mask[i]:
                try:
                    red_conv = np.power(w_global_1, -AAE_three_lambda[i])
                    green_conv = np.power(w_global_2, -AAE_three_lambda[i])
                    blue_conv = np.power(w_global_3, -AAE_three_lambda[i])
                    
                    if np.all(np.isfinite([red_conv, green_conv, blue_conv])):
                        b_abs_red[i] = AAE_constant[i] * red_conv
                        b_abs_green[i] = AAE_constant[i] * green_conv
                        b_abs_blue[i] = AAE_constant[i] * blue_conv
                except (OverflowError, ValueError):
                    pass  # Leave as NaN

# Create output dataframe - SAME LENGTH as input
file_M_3_Babs = pd.DataFrame({
    'AAE': AAE_three_lambda,
    'AAE_constant': AAE_constant,
    'sigma_AAE': sigma_AAE,
    'sigma_constant': sigma_constant,
    'b_abs_red': b_abs_red,
    'b_abs_green': b_abs_green,
    'b_abs_blue': b_abs_blue
})

# Print summary
print(f"\nPROCESSING SUMMARY:")
print(f"Total rows: {len(babs_wvl_red)}")
print(f"Successful AAE fits: {successful_fits}")
print(f"Skipped rows (filled with NaN): {skipped_rows}")
print(f"Success rate: {successful_fits/len(babs_wvl_red)*100:.1f}%")
print(f"Output dataframe shape: {file_M_3_Babs.shape}")

# Export CSV - SAME NUMBER OF ROWS as input
export_csv = file_M_3_Babs.to_csv(rf'C:\Users\haika\Desktop\May_Research\MAC Model for may dataset\ModuleC\Output_Module_C_Babs_CORA.csv', index=None, header=True)

print(f"Successfully exported CSV with {len(file_M_3_Babs)} rows (same as input)")
print(f"Non-NaN AAE values: {file_M_3_Babs['AAE'].notna().sum()}")
print(f"Non-NaN absorption values: {file_M_3_Babs['b_abs_red'].notna().sum()}")
print(f"Note: 'Successful' includes all finite input data, even if fitting failed or values were zero/negative")

Processing 1382863 rows...


C:\Users\haika\AppData\Local\Temp\ipykernel_25232\1484679383.py:85: OptimizeWarning: Covariance of the parameters could not be estimated
  W_coef, w_sigma = curve_fit(AAE_fit_one_measurement, wavelength_instrument, one_babs_at_three_lambda, p0=W_coef, maxfev=1000)
C:\Users\haika\AppData\Local\Temp\ipykernel_25232\1484679383.py:61: OptimizeWarning: Covariance of the parameters could not be estimated
  W_coef, w_sigma = curve_fit(AAE_fit_one_measurement, wavelength_instrument, one_babs_at_three_lambda, p0=W_coef, maxfev=1000)
C:\Users\haika\AppData\Local\Temp\ipykernel_25232\1380745124.py:5: RuntimeWarning: overflow encountered in power
  return  AAE_constant*np.power(wavelength, -AAE_fit)
c:\Users\haika\AppData\Local\Programs\Python\Python311\Lib\site-packages\scipy\optimize\_minpack_py.py:493: RuntimeWarning: overflow encountered in matmul
  cov_x = invR @ invR.T
C:\Users\haika\AppData\Local\Temp\ipykernel_25232\1380745124.py:5: RuntimeWarning: overflow encountered in multiply
  return

  Processed 10000/1382863 rows... (successful: 10000, skipped: 0)
  Processed 20000/1382863 rows... (successful: 20000, skipped: 0)


C:\Users\haika\AppData\Local\Temp\ipykernel_25232\1484679383.py:109: OptimizeWarning: Covariance of the parameters could not be estimated
  W_coef, w_sigma = curve_fit(AAE_fit_one_measurement, wavelength_instrument, one_babs_at_three_lambda, p0=W_coef, maxfev=1000)


  Processed 30000/1382863 rows... (successful: 30000, skipped: 0)
  Processed 40000/1382863 rows... (successful: 40000, skipped: 0)
  Processed 50000/1382863 rows... (successful: 50000, skipped: 0)
  Processed 60000/1382863 rows... (successful: 60000, skipped: 0)
  Processed 70000/1382863 rows... (successful: 70000, skipped: 0)
  Processed 80000/1382863 rows... (successful: 80000, skipped: 0)
  Processed 90000/1382863 rows... (successful: 90000, skipped: 0)
  Processed 100000/1382863 rows... (successful: 100000, skipped: 0)
  Processed 110000/1382863 rows... (successful: 110000, skipped: 0)
  Processed 120000/1382863 rows... (successful: 120000, skipped: 0)
  Processed 130000/1382863 rows... (successful: 130000, skipped: 0)
  Processed 140000/1382863 rows... (successful: 140000, skipped: 0)
  Processed 150000/1382863 rows... (successful: 150000, skipped: 0)
  Processed 160000/1382863 rows... (successful: 160000, skipped: 0)
  Processed 170000/1382863 rows... (successful: 170000, skippe

In [ ]:

file_M_3_Babs

,AAE,AAE_constant,sigma_AAE,sigma_constant,b_abs_red,b_abs_green,b_abs_blue
0,1.791211,8.278461e+05,1.170669,6.027053e+06,7.371415,10.845913,13.541224
1,0.280488,2.597778e+01,5.104331,8.341052e+02,4.204876,4.467000,4.624982
2,-2.577573,7.430706e-07,1.812989,8.712196e-06,13.759750,7.893442,5.735318
3,-1.310525,5.770382e-03,0.763381,2.823190e-02,28.594769,21.556544,18.325459
4,0.301646,5.531419e+01,1.748935,6.099235e+02,7.804296,8.328705,8.645899
...,...,...,...,...,...,...,...
1382858,-2.070441,3.741481e-06,2.077035,5.016665e-05,2.574805,1.647722,1.274870
1382859,-0.900338,-1.614026e-02,0.229645,2.366162e-02,-5.577689,-4.593604,-4.108686
1382860,1.370850,-1.780897e+04,1.879193,2.087719e+05,-2.429227,-3.264554,-3.868962
1382861,2.833662,-1.728251e+08,1.328921,1.419227e+09,-1.769993,-3.260573,-4.632164


In [ ]:
# Merge Bscat & Babs dataframe to further calculate SSA
optical_df = pd.concat([file_M_3_Bscat, file_M_3_Babs], axis=1)
optical_df

,SAE,SAE_constant,b_scat_red,b_scat_green,b_scat_blue,AAE,AAE_constant,sigma_AAE,sigma_constant,b_abs_red,b_abs_green,b_abs_blue
0,0.348673,7.656526e+01,7.960355,8.581822,8.960720,1.791211,8.278461e+05,1.170669,6.027053e+06,7.371415,10.845913,13.541224
1,2.297221,1.165917e+07,3.886437,6.377430,8.477501,0.280488,2.597778e+01,5.104331,8.341052e+02,4.204876,4.467000,4.624982
2,3.019193,1.084825e+09,3.331293,6.387145,9.284979,-2.577573,7.430706e-07,1.812989,8.712196e-06,13.759750,7.893442,5.735318
3,3.592405,3.662760e+10,2.721853,5.905135,9.216175,-1.310525,5.770382e-03,0.763381,2.823190e-02,28.594769,21.556544,18.325459
4,1.538964,2.099477e+05,9.614668,13.397724,16.212444,0.301646,5.531419e+01,1.748935,6.099235e+02,7.804296,8.328705,8.645899
...,...,...,...,...,...,...,...,...,...,...,...,...
1382858,0.811949,2.433220e+03,12.498418,14.889486,16.465456,-2.070441,3.741481e-06,2.077035,5.016665e-05,2.574805,1.647722,1.274870
1382859,0.766437,2.163734e+03,14.934809,17.618259,19.373491,-0.900338,-1.614026e-02,0.229645,2.366162e-02,-5.577689,-4.593604,-4.108686
1382860,1.410880,1.127574e+05,11.860586,16.077189,19.148506,1.370850,-1.780897e+04,1.879193,2.087719e+05,-2.429227,-3.264554,-3.868962
1382861,1.019745,9.755142e+03,13.002259,16.199435,18.381298,2.833662,-1.728251e+08,1.328921,1.419227e+09,-1.769993,-3.260573,-4.632164


### SSA calculation

In [ ]:
#replace values that could cause calculation errors with another flag (-3333)
optical_df.loc[optical_df['b_abs_red']+optical_df['b_scat_red'] == 0, 'b_abs_red'] = -3333
optical_df.loc[optical_df['b_abs_green']+optical_df['b_scat_green'] == 0, 'b_abs_green'] = -3333
optical_df.loc[optical_df['b_abs_blue']+optical_df['b_scat_blue'] == 0, 'b_abs_blue'] = -3333

optical_df.loc[optical_df['b_abs_red']== 0, 'b_scat_red'] = -3333
optical_df.loc[optical_df['b_abs_green']== 0, 'b_scat_green'] = -3333
optical_df.loc[optical_df['b_abs_blue']== 0, 'b_scat_blue'] = -3333

In [15]:
# Check
#optical_df[optical_df['b_scat_green']== -3333]

In [ ]:
#calculate SSA
optical_df['SSA_red'] = optical_df.apply(lambda x: x['b_scat_red'] / (x['b_scat_red'] + x['b_abs_red']), axis=1)
optical_df['SSA_blue'] = optical_df.apply(lambda x: x['b_scat_blue'] / (x['b_scat_blue'] + x['b_abs_blue']), axis=1)
optical_df['SSA_green'] = optical_df.apply(lambda x: x['b_scat_green'] / (x['b_scat_green'] + x['b_abs_green']), axis=1)

In [ ]:
#Check
optical_df[optical_df['SSA_red']<0]

,SAE,SAE_constant,b_scat_red,b_scat_green,b_scat_blue,AAE,AAE_constant,sigma_AAE,sigma_constant,b_abs_red,b_abs_green,b_abs_blue,SSA_red,SSA_blue,SSA_green
263,0.754644,2.304529e+02,1.717237,2.020641,2.218704,-0.000021,-6.664953e+03,2.141761e-06,9.030285e-02,-6665.860778,-6665.830632,-6665.813307,-0.000258,-0.000333,-0.000303
264,2.204175,1.581073e+06,0.964237,1.550835,2.037890,-0.000016,-6.665176e+03,1.627463e-06,6.861977e-02,-6665.865338,-6665.842442,-6665.829284,-0.000145,-0.000306,-0.000233
266,3.665500,4.537830e+09,0.209802,0.462401,0.728237,-0.000018,-6.665058e+03,1.818004e-06,7.664934e-02,-6665.827831,-6665.802254,-6665.787554,-0.000031,-0.000109,-0.000069
335,0.636116,1.421297e+02,2.286276,2.622344,2.837406,-0.000003,-3.332858e+03,3.581053e-07,7.548784e-03,-3332.933229,-3332.930720,-3332.929277,-0.000686,-0.000852,-0.000787
336,0.660396,1.506201e+02,2.069512,2.386175,2.589648,-0.000015,-4.998971e+03,1.582812e-06,5.004901e-02,-4999.474170,-4999.457467,-4999.447868,-0.000414,-0.000518,-0.000478
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1382845,11.570413,1.834137e+31,0.043666,0.529063,2.218987,2.494981,-3.191785e+07,2.012686e-01,3.978707e+07,-2.946622,-5.045861,-6.873844,-0.015042,-0.476704,-0.117132
1382847,2.333853,5.496819e+06,1.444479,2.389105,3.190280,0.818259,-4.439976e+02,6.104175e+00,1.701585e+04,-2.189084,-2.611428,-2.890092,-1.939927,10.627614,-10.746135
1382852,2.741965,8.505504e+07,1.579821,2.853281,4.007744,1.191725,-9.285012e+03,5.172354e-01,3.002843e+04,-4.051913,-5.238949,-6.072610,-0.639063,-1.940923,-1.196009
1382853,1.013256,1.932396e+03,2.686434,3.342332,3.789455,0.196830,-2.237646e+01,1.009043e+00,1.425270e+02,-6.234753,-6.505025,-6.665629,-0.757100,-1.317533,-1.056800


In [ ]:
#replace values where SSA <0 or >1 with a flag (-2222)
optical_df.loc[optical_df['SSA_red'] < 0, 'SSA_red'] = -2222
optical_df.loc[optical_df['SSA_green'] < 0, 'SSA_green'] = -2222
optical_df.loc[optical_df['SSA_blue'] < 0, 'SSA_blue'] = -2222
optical_df.loc[optical_df['SSA_red'] > 1, 'SSA_red'] = -2222
optical_df.loc[optical_df['SSA_green'] > 1, 'SSA_green'] = -2222
optical_df.loc[optical_df['SSA_blue'] > 1, 'SSA_blue'] = -2222

In [19]:
# # Why this ratio is needed?
# #calculate ratio of abs to scat
# optical_df['red_scat_abs'] = optical_df['b_scat_red']/optical_df['b_abs_red']
# optical_df['green_scat_abs'] = optical_df['b_scat_green']/optical_df['b_abs_green']
# optical_df['blue_scat_abs'] = optical_df['b_scat_blue']/optical_df['b_abs_blue']

# #replace values where ratio < 0 with a flag (-2222)
# optical_df.loc[optical_df['red_scat_abs'] < 0, 'red_scat_abs'] = -2222
# optical_df.loc[optical_df['green_scat_abs'] < 0, 'green_scat_abs'] = -2222
# optical_df.loc[optical_df['blue_scat_abs'] < 0, 'blue_scat_abs'] = -2222

In [ ]:
optical_df

,SAE,SAE_constant,b_scat_red,b_scat_green,b_scat_blue,AAE,AAE_constant,sigma_AAE,sigma_constant,b_abs_red,b_abs_green,b_abs_blue,SSA_red,SSA_blue,SSA_green
0,0.348673,7.656526e+01,7.960355,8.581822,8.960720,1.791211,8.278461e+05,1.170669,6.027053e+06,7.371415,10.845913,13.541224,0.519207,0.398220,0.441730
1,2.297221,1.165917e+07,3.886437,6.377430,8.477501,0.280488,2.597778e+01,5.104331,8.341052e+02,4.204876,4.467000,4.624982,0.480322,0.647015,0.588083
2,3.019193,1.084825e+09,3.331293,6.387145,9.284979,-2.577573,7.430706e-07,1.812989,8.712196e-06,13.759750,7.893442,5.735318,0.194915,0.618162,0.447261
3,3.592405,3.662760e+10,2.721853,5.905135,9.216175,-1.310525,5.770382e-03,0.763381,2.823190e-02,28.594769,21.556544,18.325459,0.086914,0.334627,0.215032
4,1.538964,2.099477e+05,9.614668,13.397724,16.212444,0.301646,5.531419e+01,1.748935,6.099235e+02,7.804296,8.328705,8.645899,0.551966,0.652193,0.616656
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1382858,0.811949,2.433220e+03,12.498418,14.889486,16.465456,-2.070441,3.741481e-06,2.077035,5.016665e-05,2.574805,1.647722,1.274870,0.829180,0.928137,0.900363
1382859,0.766437,2.163734e+03,14.934809,17.618259,19.373491,-0.900338,-1.614026e-02,0.229645,2.366162e-02,-5.577689,-4.593604,-4.108686,-2222.000000,-2222.000000,-2222.000000
1382860,1.410880,1.127574e+05,11.860586,16.077189,19.148506,1.370850,-1.780897e+04,1.879193,2.087719e+05,-2.429227,-3.264554,-3.868962,-2222.000000,-2222.000000,-2222.000000
1382861,1.019745,9.755142e+03,13.002259,16.199435,18.381298,2.833662,-1.728251e+08,1.328921,1.419227e+09,-1.769993,-3.260573,-4.632164,-2222.000000,-2222.000000,-2222.000000


In [ ]:
#Add datetime column at the very beginning
dateparse = lambda x: datetime.strptime(x, '%m/%d/%Y %H:%M:%S')
datetimedf = pd.read_csv(rf"C:\Users\haika\Desktop\May_Research\MAC Model for may dataset\ModuleA\datetime.csv", parse_dates=['datetime_all'], date_parser=dateparse)['datetime_all']
optical_df.insert(0, 'datetime', datetimedf)



export_csv = optical_df.to_csv(rf'C:\Users\haika\Desktop\May_Research\MAC Model for may dataset\ModuleC\Output_Module_C_Babs_Bscat_SSA_HANYANG+CORA.csv', index = None, header=True)

C:\Users\haika\AppData\Local\Temp\ipykernel_25232\1414881059.py:3: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  datetimedf = pd.read_csv(rf"C:\Users\haika\Desktop\May_Research\MAC Model for may dataset\ModuleA\datetime.csv", parse_dates=['datetime_all'], date_parser=dateparse)['datetime_all']
